<a href="https://colab.research.google.com/github/Julien-Dufresne/house_pricing/blob/Catboost/New_Sherlock_Housing_Competition_Catboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Housing Competition using Catboost

In [ ]:
!pip install catboost

In [ ]:
# Essential imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel, RFE
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

# Set a random state for reproducibility
RANDOM_STATE = 710

# 1. Pre-processing

1.1 Loading data, removing ID and assigning SalesPrices to y, splitting the data into train and test

In [ ]:
# set_config(transform_output="pandas") tells scikit-learn to return transformed features as a pandas DataFrame instead of a NumPy array.
# That is useful because the result keeps readable column names after preprocessing.
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, root_mean_squared_error, r2_score
# Import data

url = "https://docs.google.com/spreadsheets/d/1IbAAe8x9inoiLXg88fDjKV0xTN1U6onrCPHuvseNQI4/edit?gid=0#gid=0$0"

sheet_id = url.split("/d/")[1].split("/")[0]
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

# naming our dataset housing_regression_data
housing_regression_data = pd.read_csv(csv_url)

# X and y creation
X = housing_regression_data.drop(columns="Id", errors="ignore")
y = X.pop("SalePrice")



# data splitting with 20% as test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# compresses large values much more than small values, which can make some regression problems easier to model.
y_train_log = np.log1p(y_train)

## Pre processing, scaler and Pipelines

In [ ]:
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

# Make copies so the original X_train and X_test are unchanged.
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()

# Identify text/category columns. CatBoost will handle these directly,
# so we do not need one-hot encoding or the earlier scikit-learn preprocessor.
cat_cols = X_train_cb.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# MSSubClass contains numeric-looking codes, but they represent categories.
# Add it to CatBoost's categorical column list.
if "MSSubClass" not in cat_cols:
    cat_cols.append("MSSubClass")

# CatBoost requires categorical values to be strings, not missing values.
# Apply identical cleaning to train and test data.
for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].fillna("Missing").astype(str)
    X_test_cb[col] = X_test_cb[col].fillna("Missing").astype(str)


In [ ]:
# Create a CatBoost regression model.
cat_model = CatBoostRegressor(
     iterations=2000,        # Maximum number of boosting trees
    learning_rate=0.03,     # Small contribution from each tree
    depth=6,                # Complexity of each tree
    l2_leaf_reg=5,          # Regularization to reduce overfitting
    loss_function="RMSE",   # Optimizes root mean squared error
    random_seed=710,        # Makes results reproducible
    verbose=False,          # Suppresses training output during CV
    thread_count=1          # One core per fit; CV can handle parallelism
)

# Run 5-fold cross-validation.
# Each fold trains on 80% of the data and validates on the remaining 20%.
# The minus sign converts sklearn's negative RMSE score into normal RMSE.
cat_cv = -cross_val_score(
    cat_model,
    X_train_cb,
    y_train_log,                         # log1p(SalePrice)
    cv=5,
    scoring="neg_root_mean_squared_error",
    params={"cat_features": cat_cols},   # Tell CatBoost which columns are categorical
    n_jobs=-1                            # Run folds in parallel
)

print(f"CatBoost CV log-RMSE: {cat_cv.mean():.4f}")

In [ ]:
# Train one final model on every training row after evaluation.
cat_model.fit(
    X_train_cb,
    y_train_log,
    cat_features=cat_cols,
    verbose=200
)

# Predict log prices, then convert predictions back into dollar prices.
pred_log = cat_model.predict(X_test_cb)
predictions = np.expm1(pred_log)

In [ ]:
results = pd.DataFrame({
    "Id": id_column,
    "SalePrice": predictions
})

results.to_csv("submission_catboost.csv", index=False)